# Multi-Stage Direct-Guided ResNet-50 dengan 5-Fold CV

Notebook ini mengikuti alur eksperimen segmentation-guided lama, tetapi memakai model independen `MultiStageDirectGuidedResNet50`. Data dibaca dari arsip eksperimen `fa028196-fd4c-441f-a5d3-3efb9707e7f9`, training memakai lima fold yang sama, dan pengujian membuat Grad-CAM serta LRP untuk `layer1` sampai `layer4`. Progress `tqdm` ditampilkan saat copy, checksum, ekstraksi, validasi data, training, validation, holdout, dan XAI.

## Arsip yang digunakan

Notebook mengharapkan kedua file berikut sudah tersedia di Google Drive di bawah repository:

`experiment_results/fa028196-fd4c-441f-a5d3-3efb9707e7f9/fa028196-fd4c-441f-a5d3-3efb9707e7f9_guided_resnet50_parenchyma.tar.gz`

`experiment_results/fa028196-fd4c-441f-a5d3-3efb9707e7f9/fa028196-fd4c-441f-a5d3-3efb9707e7f9_guided_resnet50_parenchyma.tar.gz.sha256`

## 1. Periksa GPU

In [ ]:
import platform
import shutil
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU tidak tersedia. Aktifkan GPU pada pengaturan runtime Colab."
    )

disk = shutil.disk_usage("/content")
gpu_memory = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f"Python: {platform.python_version()}")
print(f"PyTorch: {torch.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"GPU memory: {gpu_memory:.1f} GiB")
print(f"Free disk: {disk.free / (1024**3):.1f} GiB")

## 2. Hubungkan Google Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 3. Atur konfigurasi eksperimen

Semua parameter yang biasa diubah dikumpulkan pada satu cell. Nilai training, optimizer, augmentasi, early stopping, dan split mengikuti eksperimen pembanding.

In [ ]:
from pathlib import Path

# Repository
REPOSITORY_URL = (
    "https://github.com/FillipusAditya/"
    "mask-guided-lung-nodule-xai.git"
)
REPOSITORY_BRANCH = "refactor/002-segmentation"
PROJECT_ROOT = Path("/content/mask-guided-lung-nodule-xai")

# Google Drive dan eksperimen
DRIVE_PROJECT_ROOT = Path(
    "/content/drive/MyDrive/mask-guided-lung-nodule-xai"
)
EXPERIMENT_ID = "fa028196-fd4c-441f-a5d3-3efb9707e7f9"
EXPERIMENT_COMPONENT = (
    "classification/multistage_direct_guided_cv_resnet50"
)

# Arsip input
ARCHIVE_NAME = f"{EXPERIMENT_ID}_guided_resnet50_parenchyma.tar.gz"
DRIVE_EXPERIMENT_ROOT = (
    DRIVE_PROJECT_ROOT / "experiment_results" / EXPERIMENT_ID
)
DRIVE_ARCHIVE_PATH = DRIVE_EXPERIMENT_ROOT / ARCHIVE_NAME
DRIVE_CHECKSUM_PATH = DRIVE_EXPERIMENT_ROOT / f"{ARCHIVE_NAME}.sha256"
LOCAL_ARCHIVE_PATH = Path("/content") / ARCHIVE_NAME
EXTRACTION_ROOT = (
    Path("/content")
    / f"direct_guided_training_data_{EXPERIMENT_ID}"
)
COPY_ARCHIVE_TO_LOCAL = True
VERIFY_ARCHIVE_SHA256 = True
FORCE_EXTRACT = False

# Path setelah ekstraksi
DATASET_ROOT = EXTRACTION_ROOT / "000_dataset_v2/_segmentation_dataset"
METADATA_FILENAME = "004_classification_cv_5fold_seed42.csv"
PROBABILITY_ROOT = (
    EXTRACTION_ROOT
    / "experiment_results"
    / EXPERIMENT_ID
    / "segmentation/unet/inference/probability_npy"
)

# Output persisten
DRIVE_OUTPUT_ROOT = DRIVE_PROJECT_ROOT / "experiment_results"
DRIVE_OUTPUT_DIR = (
    DRIVE_OUTPUT_ROOT / EXPERIMENT_ID / EXPERIMENT_COMPONENT
)

# Input dan split
CT_INPUT_TYPE = "parenchyma"
CT_PATH_COLUMN = "ct_parenchyma_path"
INPUT_HEIGHT = 224
INPUT_WIDTH = 224
NUM_FOLDS = 5
CLASS_TO_IDX = {"benign": 0, "malignant": 1}
NORMALIZATION_MEAN = [0.485, 0.456, 0.406]
NORMALIZATION_STD = [0.229, 0.224, 0.225]

# Model direct-guided
PRETRAINED_WEIGHTS = "DEFAULT"
CLASSIFIER_DROPOUT = 0.3
ALPHA_LAYER1 = 1.0
ALPHA_LAYER2 = 1.0
ALPHA_LAYER3 = 1.0
LEARNABLE_ALPHA = False
GUIDANCE_RESIZE_MODE = "max"

# Training
NUM_EPOCHS = 100
BATCH_SIZE = 32  # Kurangi jika GPU kehabisan memori.
LEARNING_RATE = 1e-3
SEED = 42
TRANSFORM_SEED = 42
CLASSIFICATION_THRESHOLD = 0.5
DEVICE = "auto"
AMP_ENABLED = False

# Optimizer SGD
OPTIMIZER_NAME = "SGD"
MOMENTUM = 0.9
WEIGHT_DECAY = 1e-4
NESTEROV = False

# Early stopping dan pemilihan model
EARLY_STOPPING_ENABLED = True
EARLY_STOPPING_MONITOR = "val_loss"
EARLY_STOPPING_MODE = "min"
EARLY_STOPPING_PATIENCE = 20
EARLY_STOPPING_MIN_DELTA = 0.0
RESTORE_BEST_WEIGHTS = True
SAVE_LATEST_CHECKPOINT = True

# DataLoader
NUM_WORKERS = 2
PERSISTENT_WORKERS = True
PREFETCH_FACTOR = 2
PIN_MEMORY = True
TRAIN_SHUFFLE = True
VAL_SHUFFLE = False
TRAIN_DROP_LAST = False
VAL_DROP_LAST = False

# Holdout dan XAI
RUN_TEST_AFTER_TRAINING = True
MAX_TEST_SAMPLES = None  # None = seluruh holdout; 8 = smoke XAI.
TEST_BATCH_SIZE = 1
TEST_NUM_WORKERS = 0
TEST_DPI = 120
FAIL_IF_TEST_OUTPUT_EXISTS = True

print(f"Archive: {DRIVE_ARCHIVE_PATH}")
print(f"Checksum: {DRIVE_CHECKSUM_PATH}")
print(f"Output: {DRIVE_OUTPUT_DIR}")

## 4. Clone atau perbarui repository

In [ ]:
import subprocess

if (PROJECT_ROOT / ".git").is_dir():
    subprocess.run(
        ["git", "-C", str(PROJECT_ROOT), "fetch", "origin"],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(PROJECT_ROOT), "checkout", REPOSITORY_BRANCH],
        check=True,
    )
    subprocess.run(
        [
            "git", "-C", str(PROJECT_ROOT), "pull", "--ff-only",
            "origin", REPOSITORY_BRANCH,
        ],
        check=True,
    )
else:
    subprocess.run(
        [
            "git", "clone", "--branch", REPOSITORY_BRANCH,
            "--single-branch", REPOSITORY_URL, str(PROJECT_ROOT),
        ],
        check=True,
    )

training_script = (
    PROJECT_ROOT
    / "003_classification/multistage_direct_guided_cv_resnet50/train.py"
)
test_script = training_script.with_name("test.py")
for script in (training_script, test_script):
    if not script.is_file():
        raise FileNotFoundError(f"Script tidak ditemukan: {script}")
print(f"Repository siap: {PROJECT_ROOT}")

## 5. Instal package yang dibutuhkan

In [ ]:
import sys

packages = [
    "albumentations>=2.0,<3.0",
    "opencv-python-headless",
    "pandas",
    "matplotlib",
    "scikit-learn",
    "tqdm",
    "zennit==0.5.1",
]
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", *packages],
    check=True,
)
from tqdm.auto import tqdm
print("Dependencies siap, termasuk tqdm dan Zennit.")

## 6. Salin, verifikasi SHA-256, dan ekstrak arsip

In [ ]:
import hashlib
import hmac
import tarfile

def copy_file_with_progress(source, destination, chunk_size=8 * 1024 * 1024):
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = destination.with_suffix(destination.suffix + ".part")
    with source.open("rb") as input_file, temporary_path.open("wb") as output_file:
        with tqdm(total=source.stat().st_size, desc="Copy archive", unit="B", unit_scale=True) as bar:
            while True:
                chunk = input_file.read(chunk_size)
                if not chunk:
                    break
                output_file.write(chunk)
                bar.update(len(chunk))
    temporary_path.replace(destination)

def sha256_with_progress(source, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with source.open("rb") as input_file:
        with tqdm(total=source.stat().st_size, desc="SHA-256", unit="B", unit_scale=True) as bar:
            while True:
                chunk = input_file.read(chunk_size)
                if not chunk:
                    break
                digest.update(chunk)
                bar.update(len(chunk))
    return digest.hexdigest()

if not DRIVE_ARCHIVE_PATH.is_file():
    raise FileNotFoundError(f"Arsip tidak ditemukan: {DRIVE_ARCHIVE_PATH}")
if not DRIVE_CHECKSUM_PATH.is_file():
    raise FileNotFoundError(f"Checksum tidak ditemukan: {DRIVE_CHECKSUM_PATH}")

if COPY_ARCHIVE_TO_LOCAL:
    archive_path = LOCAL_ARCHIVE_PATH
    current = (
        archive_path.is_file()
        and archive_path.stat().st_size == DRIVE_ARCHIVE_PATH.stat().st_size
    )
    if not current:
        copy_file_with_progress(DRIVE_ARCHIVE_PATH, archive_path)
else:
    archive_path = DRIVE_ARCHIVE_PATH

archive_sha256 = None
if VERIFY_ARCHIVE_SHA256:
    expected_sha256 = DRIVE_CHECKSUM_PATH.read_text(encoding="utf-8").split()[0].lower()
    archive_sha256 = sha256_with_progress(archive_path)
    if not hmac.compare_digest(archive_sha256, expected_sha256):
        raise RuntimeError("Checksum arsip tidak cocok.")
    print(f"SHA-256 valid: {archive_sha256}")

marker_id = archive_sha256[:16] if archive_sha256 else str(archive_path.stat().st_size)
extraction_marker = EXTRACTION_ROOT / f".complete_{marker_id}"
if FORCE_EXTRACT and EXTRACTION_ROOT.exists():
    shutil.rmtree(EXTRACTION_ROOT)

if extraction_marker.is_file():
    print(f"Menggunakan hasil ekstraksi: {EXTRACTION_ROOT}")
else:
    EXTRACTION_ROOT.mkdir(parents=True, exist_ok=True)
    with tarfile.open(archive_path, mode="r:gz") as archive:
        members = archive.getmembers()
        unpacked_bytes = sum(m.size for m in members if m.isfile())
        free_bytes = shutil.disk_usage(EXTRACTION_ROOT.parent).free
        if free_bytes < int(unpacked_bytes * 1.10):
            raise RuntimeError("Ruang disk lokal tidak cukup untuk ekstraksi.")
        for member in tqdm(members, desc="Extract archive", unit="file"):
            archive.extract(member, path=EXTRACTION_ROOT, filter="data")
    extraction_marker.touch()
print(f"Data siap: {EXTRACTION_ROOT}")

## 7. Validasi CT, mask, probability map, dan split

In [ ]:
import csv

metadata_path = DATASET_ROOT / METADATA_FILENAME
required_columns = {
    "dataset", "patient_id", "filename", CT_PATH_COLUMN,
    "mask_path", "label", "cv_group_id", "cv_nodule_id",
    "cv_role", "cv_fold",
}
with metadata_path.open("r", encoding="utf-8", newline="") as file:
    reader = csv.DictReader(file)
    columns = set(reader.fieldnames or [])
    rows = list(reader)
missing_columns = required_columns - columns
if missing_columns:
    raise ValueError(f"Kolom metadata kurang: {sorted(missing_columns)}")

missing_files = []
for row in tqdm(rows, desc="Validate paired data", unit="sample"):
    for path in (
        DATASET_ROOT / row[CT_PATH_COLUMN],
        DATASET_ROOT / row["mask_path"],
        PROBABILITY_ROOT / Path(row["filename"]).name,
    ):
        if not path.is_file():
            missing_files.append(path)
if missing_files:
    raise FileNotFoundError(
        f"Ada {len(missing_files)} file hilang; contoh: {missing_files[:5]}"
    )

development_folds = {
    int(row["cv_fold"])
    for row in rows
    if row["cv_role"].strip().lower() == "development"
}
if development_folds != set(range(NUM_FOLDS)):
    raise ValueError(f"Fold tidak sesuai: {sorted(development_folds)}")
print(f"Samples: {len(rows):,}")
print(f"Development folds: {sorted(development_folds)}")
print("Semua CT, mask, dan probability map tersedia.")

## 8. Buat dan simpan konfigurasi JSON

In [ ]:
import json

if CT_INPUT_TYPE != "parenchyma" or CT_PATH_COLUMN != "ct_parenchyma_path":
    raise ValueError("Arsip ini harus memakai CT parenchyma.")
if GUIDANCE_RESIZE_MODE not in {"max", "bilinear"}:
    raise ValueError("GUIDANCE_RESIZE_MODE tidak valid.")
if any(alpha < 0 for alpha in (ALPHA_LAYER1, ALPHA_LAYER2, ALPHA_LAYER3)):
    raise ValueError("Semua alpha harus non-negative.")

config = {
    "experiment": {
        "id": EXPERIMENT_ID,
        "component": EXPERIMENT_COMPONENT,
    },
    "output": {
        "root_directory": str(DRIVE_OUTPUT_ROOT),
        "config_snapshot_filename": "multistage_direct_guided_cv_resnet50.json",
    },
    "data": {
        "dataset_root": str(DATASET_ROOT),
        "metadata_path": str(metadata_path),
        "ct_input_type": CT_INPUT_TYPE,
        "ct_path_column": CT_PATH_COLUMN,
        "probability_root": str(PROBABILITY_ROOT),
        "input_height": INPUT_HEIGHT,
        "input_width": INPUT_WIDTH,
        "class_to_idx": CLASS_TO_IDX,
        "normalization_mean": NORMALIZATION_MEAN,
        "normalization_std": NORMALIZATION_STD,
    },
    "cross_validation": {
        "num_folds": NUM_FOLDS,
        "development_role": "development",
        "holdout_role": "holdout_test",
        "holdout_fold": -1,
        "group_column": "cv_group_id",
        "nodule_column": "cv_nodule_id",
        "fold_column": "cv_fold",
        "role_column": "cv_role",
    },
    "model": {
        "architecture": "MultiStageDirectGuidedResNet50",
        "pretrained_weights": PRETRAINED_WEIGHTS,
        "training_strategy": "full_fine_tuning",
        "trainable_component": "entire_model",
        "classifier_dropout": CLASSIFIER_DROPOUT,
        "alpha_layer1": ALPHA_LAYER1,
        "alpha_layer2": ALPHA_LAYER2,
        "alpha_layer3": ALPHA_LAYER3,
        "learnable_alpha": LEARNABLE_ALPHA,
        "guidance_resize_mode": GUIDANCE_RESIZE_MODE,
    },
    "training": {
        "num_epochs": NUM_EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "seed": SEED,
        "transform_seed": TRANSFORM_SEED,
        "classification_threshold": CLASSIFICATION_THRESHOLD,
        "device": DEVICE,
    },
    "optimizer": {
        "name": OPTIMIZER_NAME,
        "momentum": MOMENTUM,
        "weight_decay": WEIGHT_DECAY,
        "nesterov": NESTEROV,
    },
    "dataloader": {
        "num_workers": NUM_WORKERS,
        "persistent_workers": PERSISTENT_WORKERS,
        "prefetch_factor": PREFETCH_FACTOR,
        "pin_memory": PIN_MEMORY,
        "train_shuffle": TRAIN_SHUFFLE,
        "val_shuffle": VAL_SHUFFLE,
        "train_drop_last": TRAIN_DROP_LAST,
        "val_drop_last": VAL_DROP_LAST,
    },
    "early_stopping": {
        "enabled": EARLY_STOPPING_ENABLED,
        "monitor": EARLY_STOPPING_MONITOR,
        "mode": EARLY_STOPPING_MODE,
        "patience": EARLY_STOPPING_PATIENCE,
        "min_delta": EARLY_STOPPING_MIN_DELTA,
        "verbose": True,
        "restore_best_weights": RESTORE_BEST_WEIGHTS,
    },
    "amp": {"training_enabled": AMP_ENABLED},
    "checkpoint": {"save_latest": SAVE_LATEST_CHECKPOINT},
}

def save_json(values, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as file:
        json.dump(values, file, indent=4)
        file.write("\n")

config_path = (
    PROJECT_ROOT
    / "003_classification/configs/multistage_direct_guided_cv_resnet50.json"
)
drive_config_path = (
    DRIVE_OUTPUT_ROOT / "saved_configs" / EXPERIMENT_ID
    / "multistage_direct_guided_cv_resnet50.json"
)
save_json(config, config_path)
save_json(config, drive_config_path)
print(json.dumps(config, indent=2))

## 9. Preflight model dan satu mini-batch

In [ ]:
import importlib

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
importlib.invalidate_caches()
package = "003_classification.multistage_direct_guided_cv_resnet50"
dataset_module = importlib.import_module(f"{package}.dataset")
model_module = importlib.import_module(f"{package}.direct_guided_resnet50")
transform_module = importlib.import_module(f"{package}.transforms")

transform = transform_module.build_evaluation_transform(
    INPUT_HEIGHT, INPUT_WIDTH, tuple(NORMALIZATION_MEAN),
    tuple(NORMALIZATION_STD), TRANSFORM_SEED,
)
preview_dataset = dataset_module.DirectGuidedClassificationDataset(
    root_dir=DATASET_ROOT, metadata_path=metadata_path, split="val",
    cv_fold=0, probability_root=PROBABILITY_ROOT, transform=transform,
    class_to_idx=CLASS_TO_IDX, ct_path_column=CT_PATH_COLUMN,
)
sample_ct, sample_probability, sample_label = preview_dataset[0]
model = model_module.MultiStageDirectGuidedResNet50(
    num_classes=2, dropout=CLASSIFIER_DROPOUT, weights=None,
    alpha_layer1=ALPHA_LAYER1, alpha_layer2=ALPHA_LAYER2,
    alpha_layer3=ALPHA_LAYER3, learnable_alpha=LEARNABLE_ALPHA,
    guidance_resize_mode=GUIDANCE_RESIZE_MODE,
).cuda().eval()
with torch.no_grad():
    logits, guidance = model(
        sample_ct.unsqueeze(0).cuda(),
        sample_probability.unsqueeze(0).cuda(),
        return_guidance_maps=True,
    )
print("CT:", tuple(sample_ct.shape))
print("Probability:", tuple(sample_probability.shape))
print("Label:", int(sample_label))
print("Logits:", tuple(logits.shape))
for name, values in guidance.items():
    print(name, tuple(values.shape))
del model
torch.cuda.empty_cache()

## 10. Training 5-fold dengan tqdm

In [ ]:
import runpy

if DRIVE_OUTPUT_DIR.exists() and any(DRIVE_OUTPUT_DIR.iterdir()):
    raise FileExistsError(
        f"Output sudah berisi artefak: {DRIVE_OUTPUT_DIR}"
    )
print(f"Mulai training: {DRIVE_OUTPUT_DIR}", flush=True)
original_arguments = sys.argv.copy()
try:
    sys.argv = [str(training_script), "--config", str(config_path)]
    runpy.run_module(
        "003_classification.multistage_direct_guided_cv_resnet50.train",
        run_name="__main__",
    )
finally:
    sys.argv = original_arguments

## 11. Validasi artefak cross-validation dan holdout

In [ ]:
import pandas as pd
from IPython.display import display

summary_path = DRIVE_OUTPUT_DIR / "cross_validation_summary.csv"
snapshot_path = DRIVE_OUTPUT_DIR / "multistage_direct_guided_cv_resnet50.json"
required = [
    summary_path, snapshot_path, DRIVE_OUTPUT_DIR / "run_config.json",
    DRIVE_OUTPUT_DIR / "holdout_predictions.csv",
    DRIVE_OUTPUT_DIR / "holdout_metrics.json",
]
for fold in range(NUM_FOLDS):
    required.extend([
        DRIVE_OUTPUT_DIR / f"fold_{fold}/best_model.pt",
        DRIVE_OUTPUT_DIR / f"fold_{fold}/latest_checkpoint.pt",
        DRIVE_OUTPUT_DIR / f"fold_{fold}/history.csv",
    ])
missing = [path for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError(f"Artefak training belum lengkap: {missing[:10]}")
summary = pd.read_csv(summary_path)
if set(summary["fold"].astype(int)) != set(range(NUM_FOLDS)):
    raise RuntimeError("Ringkasan tidak mencakup lima fold.")
display(summary)
print("Training dan holdout ensemble lengkap.")

## 12. Holdout test, Grad-CAM, LRP, dan visualisasi

Setiap sampel menghasilkan Grad-CAM dan LRP untuk `layer1`, `layer2`, `layer3`, dan `layer4`. Karena LRP empat stage untuk lima fold cukup berat, gunakan `MAX_TEST_SAMPLES=8` hanya untuk smoke XAI; gunakan `None` untuk hasil final.

In [ ]:
if RUN_TEST_AFTER_TRAINING:
    import gc

    test_output_dir = DRIVE_OUTPUT_DIR / "test"
    if (
        FAIL_IF_TEST_OUTPUT_EXISTS
        and test_output_dir.is_dir()
        and any(test_output_dir.iterdir())
    ):
        raise FileExistsError(f"Output test sudah ada: {test_output_dir}")

    module_name = (
        "003_classification.multistage_direct_guided_cv_resnet50.test"
    )
    arguments = [
        module_name, str(DRIVE_OUTPUT_DIR),
        "--dataset-root", str(DATASET_ROOT),
        "--metadata-path", str(metadata_path),
        "--probability-root", str(PROBABILITY_ROOT),
        "--batch-size", str(TEST_BATCH_SIZE),
        "--num-workers", str(TEST_NUM_WORKERS),
        "--device", DEVICE, "--dpi", str(TEST_DPI),
    ]
    if MAX_TEST_SAMPLES is not None:
        arguments.extend(["--max-samples", str(MAX_TEST_SAMPLES)])

    gc.collect()
    torch.cuda.empty_cache()
    original_arguments = sys.argv.copy()
    try:
        sys.argv = arguments
        runpy.run_module(module_name, run_name="__main__")
    finally:
        sys.argv = original_arguments
        gc.collect()
        torch.cuda.empty_cache()

    required_test_outputs = [
        test_output_dir / "test_predictions.csv",
        test_output_dir / "test_results.json",
        test_output_dir / "confusion_matrix.png",
        test_output_dir / "roc_curve.png",
        test_output_dir / "visualization",
    ]
    for stage in ("layer1", "layer2", "layer3", "layer4"):
        required_test_outputs.extend([
            test_output_dir / "gradcam_npy" / stage,
            test_output_dir / "lrp_npy" / stage,
        ])
    missing = [path for path in required_test_outputs if not path.exists()]
    if missing:
        raise RuntimeError(f"Artefak XAI belum lengkap: {missing}")
    print(f"Holdout dan XAI selesai: {test_output_dir}")
else:
    print("Holdout XAI dilewati.")

## Menjalankan ulang

Training tidak menimpa output yang sudah ada. Untuk eksperimen baru, ubah `EXPERIMENT_ID` dan arsip yang sesuai. Jika hanya test/XAI yang perlu diulang, pindahkan folder `test/` lama terlebih dahulu agar hasil parsial tetap dapat diaudit.